# OpenClaw → ComfyUI → Wan 2.2 TI2V 5B

**HP only:** buka notebook ini di Google Colab, pilih GPU, lalu jalankan dari atas ke bawah.

Target: iPhone → OpenClaw → public ComfyUI → Wan 2.2 → video.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt
# ComfyUI's requirements can replace Colab's CUDA-enabled PyTorch with a CPU build.
# Reinstall an official CUDA wheel explicitly so torch.cuda is available.
!pip -q install --upgrade --force-reinstall --no-cache-dir torch==2.10.0 --index-url https://download.pytorch.org/whl/cu126
!pip -q install requests websocket-client
import torch
print('PyTorch:',torch.__version__)
print('Torch CUDA build:',torch.version.cuda)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak aktif. Pastikan Runtime Colab memakai GPU, lalu jalankan ulang dari atas.')
print('GPU:',torch.cuda.get_device_name(0))

## Download Wan 2.2 TI2V 5B

In [ ]:
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae
!wget -q --show-progress -O models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
!wget -q --show-progress -O models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q --show-progress -O models/vae/wan2.2_vae.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors
!ls -lh models/diffusion_models models/text_encoders models/vae

## Start ComfyUI — fixed diagnostics
The old cell hid the real startup error because it sent ComfyUI output into a pipe. This version writes a log, waits for port 8188, and prints the real error if startup fails.

In [ ]:
import subprocess, time, requests, os, signal
%cd /content/ComfyUI
log_path='/content/comfyui.log'
if os.path.exists(log_path): os.remove(log_path)
with open(log_path,'w') as log:
    comfy = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--enable-cors-header','*'], stdout=log, stderr=subprocess.STDOUT, text=True)
print('ComfyUI PID:', comfy.pid)
ready=False
for i in range(120):
    if comfy.poll() is not None:
        break
    try:
        r=requests.get('http://127.0.0.1:8188/system_stats',timeout=2)
        if r.ok:
            ready=True
            print('COMFYUI READY — HTTP',r.status_code)
            break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    print('COMFYUI GAGAL START. LOG TERAKHIR:')
    print(open(log_path,errors='ignore').read()[-12000:])
    raise RuntimeError('ComfyUI tidak berhasil membuka port 8188; lihat log di atas.')

## Public URL for OpenClaw
Cloudflare Quick Tunnel akan meneruskan URL publik ke ComfyUI.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, time, re, os
cf_log='/content/cloudflared.log'
if os.path.exists(cf_log): os.remove(cf_log)
with open(cf_log,'w') as log:
    cf=subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8188'],stdout=log,stderr=subprocess.STDOUT,text=True)
url=None
for _ in range(90):
    txt=open(cf_log,errors='ignore').read()
    m=re.search(r'https://[A-Za-z0-9.-]+\.trycloudflare\.com',txt)
    if m:
        url=m.group(0)
        break
    if cf.poll() is not None:
        break
    time.sleep(1)
print('COMFYUI_BASE_URL =',url)
if not url:
    print(open(cf_log,errors='ignore').read()[-12000:])
    raise RuntimeError('Cloudflare tunnel gagal dibuat.')
open('/content/COMFYUI_BASE_URL.txt','w').write(url)

## Test endpoint

In [ ]:
import requests
base=open('/content/COMFYUI_BASE_URL.txt').read().strip()
print('BASE:',base)
r=requests.get(base+'/system_stats',timeout=30)
print('HTTP',r.status_code)
print(r.text[:1000])

## Recommended first generation
Start with portrait 480×832, 49 frames, 16 fps. Once the pipeline is stable, increase duration/quality. Free GPU runtimes still have session/quota limits.